# 1.5 — `retriever.py` (experiment)

Scratch notebook for module 1.5, per CLAUDE.md §5 step 1. Build the retrieval logic
inline against the **real** pgvector store, settle the open parameter choices, then port
the frozen result into `app/retrieval/retriever.py`.

**Prerequisite:** ingestion has run (`python -m app.retrieval.ingestion`) — 64 chunks
across 4 collections.

### Questions this notebook settles

| # | Question | Settled in |
|---|---|---|
| (a) | Is `TOP_K=4` right? The mutual-funds NAV clause ranks below it | §3 |
| (b) | Add a distance threshold, or top-k only? | §4 |
| (c) | Caching shape for the per-collection drift probe | §5 |
| (d) | How should the guard probe the store — embed a query, or read metadata? | §5b |

(a)–(c) came from the tracker. **(d) was found during the port**: a first attempt at
`app/retrieval/retriever.py` changed which exception escaped the drift guard while this
notebook still passed, which is exactly the class of thing the notebook-first workflow is
supposed to catch *before* code lands. It is written up in §5b.

### What 1.5 delivers

1. **Collection-scoped top-k search** — `retrieve(query, product_id)`, collection resolved
   from the registry (never hardcoded — invariant I-8), metadata passed through intact.
2. **`assert_store_matches` startup drift guard** — the sole enforcement point for
   **invariant I-11**: the store can never be queried with an embedding model other than
   the one that built it. Prototype is cell 27 of `1_4_ingestion.ipynb`; §5b is the
   version that ports.

## 1. Setup — real store, real registry, real settings

In [2]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from app.config.settings import settings
from app.core.registry import get_registry

registry = get_registry()
print("repo root      :", REPO_ROOT)
print("embedding model:", settings.EMBEDDING_MODEL)
print("chunking       :", settings.CHUNK_SIZE, "/", settings.CHUNK_OVERLAP)
print("TOP_K          :", settings.TOP_K)
for pid in registry.product_ids():
    print(f"  {pid:14} -> {registry.get(pid).doc_collection}")

repo root      : c:\Users\DELL\OneDrive - scoptanalytics.com\Documents\GitHub\finspring_ai_chatbot
embedding model: text-embedding-3-small
chunking       : 1000 / 150
TOP_K          : 4
  digital_fd     -> fd_docs
  digital_gold   -> gold_docs
  bonds          -> bond_docs
  mutual_funds   -> mf_docs


In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_postgres import PGVector

# One embeddings client for the whole notebook. Note this is the *query-side* embedder:
# it must be the same model that built the store, which is exactly what the §5 drift
# guard exists to enforce.
embeddings = OpenAIEmbeddings(model=settings.EMBEDDING_MODEL)


def open_store(product_id: str) -> PGVector:
    """Open one product's collection. Collection name comes from the registry (I-8)."""
    return PGVector(
        embeddings=embeddings,
        collection_name=registry.get(product_id).doc_collection,
        connection=settings.DATABASE_URL,
    )


for pid in registry.product_ids():
    n = len(open_store(pid).similarity_search("a", k=200))
    print(f"{pid:14} {registry.get(pid).doc_collection:10} {n:3} chunks")

digital_fd     fd_docs     16 chunks
digital_gold   gold_docs   16 chunks
bonds          bond_docs   19 chunks
mutual_funds   mf_docs     13 chunks


## 2. A probe set with known answers

To answer (a) and (b) with measurement rather than impression, each query is paired with
a substring that **must** appear in the retrieved context for the question to be
answerable. That turns "do the results look good?" into recall@k.

Substrings are deliberately drawn from the answer sentence itself, not the section
heading — retrieving a section *about* the topic is not the same as retrieving the fact.

In [4]:
# (product_id, query, substring that must appear in the retrieved context)
PROBES = [
    ("digital_fd",   "What is the penalty for premature withdrawal of a fixed deposit?", "Premature withdrawal"),
    ("digital_fd",   "What FD tenures are available?",                                  "7 days"),
    ("digital_fd",   "How is FD interest taxed?",                                       "Taxation of FD"),
    ("digital_gold", "How is the gold price determined at purchase?",                   "Live price"),
    ("digital_gold", "What are the storage charges for digital gold?",                  "storage"),
    ("bonds",        "What is the minimum investment amount for bonds?",                "Minimum ticket"),
    ("bonds",        "How is bond interest taxed?",                                     "Taxation"),
    ("mutual_funds", "How is NAV calculated?",                                          "units outstanding"),
    ("mutual_funds", "What is the expense ratio cap?",                                  "Total Expense Ratio"),
]

stores = {pid: open_store(pid) for pid in registry.product_ids()}
print(f"{len(PROBES)} probes across {len({p for p, _, _ in PROBES})} products")

9 probes across 4 products


## 3. Question (a) — is `TOP_K=4` right?

Sweep k and measure how many probes retrieve their answer-bearing chunk.

In [5]:
for k in (3, 4, 5, 6, 8):
    hits = sum(
        any(need.lower() in d.page_content.lower() for d in stores[pid].similarity_search(q, k=k))
        for pid, q, need in PROBES
    )
    print(f"k={k}: {hits}/{len(PROBES)} probes retrieve the answer-bearing chunk")

k=3: 8/9 probes retrieve the answer-bearing chunk
k=4: 8/9 probes retrieve the answer-bearing chunk
k=5: 9/9 probes retrieve the answer-bearing chunk
k=6: 9/9 probes retrieve the answer-bearing chunk
k=8: 9/9 probes retrieve the answer-bearing chunk


In [5]:
# Where does the answer chunk actually rank? The aggregate above hides which probe fails.
for pid, q, need in PROBES:
    ranks = [
        i for i, d in enumerate(stores[pid].similarity_search(q, k=13), 1)
        if need.lower() in d.page_content.lower()
    ]
    rank = ranks[0] if ranks else "NOT FOUND"
    flag = "  <-- misses at k=4" if (not ranks or ranks[0] > 4) else ""
    print(f"rank {str(rank):>9}  {pid:13} {q[:48]:50}{flag}")

rank         1  digital_fd    What is the penalty for premature withdrawal of   


rank         3  digital_fd    What FD tenures are available?                    


rank         1  digital_fd    How is FD interest taxed?                         


rank         1  digital_gold  How is the gold price determined at purchase?     


rank         3  digital_gold  What are the storage charges for digital gold?    


rank         1  bonds         What is the minimum investment amount for bonds?  


rank         1  bonds         How is bond interest taxed?                       


rank         5  mutual_funds  How is NAV calculated?                              <-- misses at k=4


rank         1  mutual_funds  What is the expense ratio cap?                    


### Finding (a) — keep `TOP_K=4`

```
k=3: 8/9    k=4: 8/9    k=5: 9/9    k=6: 9/9    k=8: 9/9
```

k=5 reaching 9/9 looks like an argument for raising k. It is not. Eight of the nine probes
put their answer at **rank ≤ 3** — the single probe k=5 rescues is
`mutual_funds :: How is NAV calculated?` at rank 5, which is the **already-documented KB
content gap** (tracker, Stage 1): the NAV formula exists only as a subordinate clause
inside chunk 2, headed `1. What is a mutual fund?`, whose embedding is dominated by fund
structure and SEBI regulation.

So raising k to 5 would buy one point of recall by dragging in a chunk that is topically
wrong but happens to contain the sentence, while adding a fifth chunk of noise to the
eight probes already satisfied at rank ≤ 3. That is tuning retrieval to paper over a
documentation defect — and it degrades `generate`, which must reason over everything
handed to it.

**Decision: `TOP_K=4` unchanged.** The NAV gap stays a doc fix (add a numbered NAV section,
re-ingest `--product mutual_funds`), exactly as the tracker records. Until then the
architecture behaves correctly: no context → `INSUFFICIENT_CONTEXT` → `offer_ticket`.

## 4. Question (b) — distance threshold, or top-k only?

The appeal of a threshold is dropping junk before it reaches `generate`. For that to work,
correct chunks and wrong chunks have to be **separable by absolute distance**. Test that
directly: compare the distance of each probe's answer chunk against the best *wrong* chunk
returned for the same query.

In [6]:
answer_scores, distractor_scores = [], []

print(f"{'product':14} {'answer':>7} {'best wrong':>11}   query")
for pid, q, need in PROBES:
    scored = stores[pid].similarity_search_with_score(q, k=8)
    ans = [s for d, s in scored if need.lower() in d.page_content.lower()]
    wrong = [s for d, s in scored if need.lower() not in d.page_content.lower()]
    if ans:
        answer_scores.append(min(ans))
    distractor_scores.append(min(wrong))
    a = f"{min(ans):.3f}" if ans else "  --"
    print(f"{pid:14} {a:>7} {min(wrong):>11.3f}   {q[:44]}")

print(f"\ncorrect chunks   : {min(answer_scores):.3f} - {max(answer_scores):.3f}")
print(f"top distractors  : {min(distractor_scores):.3f} - {max(distractor_scores):.3f}")
print(f"OVERLAP          : {max(min(answer_scores), min(distractor_scores)):.3f} "
      f"- {min(max(answer_scores), max(distractor_scores)):.3f}")

product         answer  best wrong   query


digital_fd       0.332       0.528   What is the penalty for premature withdrawal


digital_fd       0.478       0.437   What FD tenures are available?


digital_fd       0.332       0.401   How is FD interest taxed?


digital_gold     0.409       0.475   How is the gold price determined at purchase


digital_gold     0.377       0.335   What are the storage charges for digital gol


bonds            0.428       0.500   What is the minimum investment amount for bo


bonds            0.432       0.479   How is bond interest taxed?


mutual_funds     0.709       0.585   How is NAV calculated?


mutual_funds     0.516       0.666   What is the expense ratio cap?

correct chunks   : 0.332 - 0.709
top distractors  : 0.335 - 0.666
OVERLAP          : 0.335 - 0.666


In [7]:
# What would a threshold actually cost? Sweep candidate cutoffs over the probe set.
print(f"{'cutoff':>7}  {'answers kept':>13}  {'distractors cut':>16}")
for cutoff in (0.45, 0.50, 0.55, 0.60, 0.65, 0.70):
    kept = sum(s <= cutoff for s in answer_scores)
    cut = sum(s > cutoff for s in distractor_scores)
    print(f"{cutoff:>7.2f}  {kept:>6}/{len(answer_scores):<6}  {cut:>9}/{len(distractor_scores):<6}")

 cutoff   answers kept   distractors cut
   0.45       6/9               6/9     
   0.50       7/9               4/9     
   0.55       8/9               2/9     
   0.60       8/9               1/9     
   0.65       8/9               1/9     
   0.70       8/9               0/9     


### Finding (b) — top-k only, no threshold

The measured result is worse than "the ranges overlap." In **3 of 9 probes the correct
chunk scores *further away* than the best distractor**:

| probe | answer | best distractor |
|---|---|---|
| FD tenures | 0.478 | **0.437** |
| gold storage charges | 0.377 | **0.335** |
| NAV calculation | 0.709 | **0.585** |

Correct chunks span 0.332–0.709; top distractors span 0.335–0.666. The bands are almost
entirely coincident, so distance carries **no separating signal at all** on this corpus.

The cutoff sweep confirms there is no good operating point — every cutoff trades roughly
one correct answer for one distractor:

```
cutoff   answers kept   distractors cut
  0.45       6/9              6/9
  0.50       7/9              4/9
  0.55       8/9              2/9
  0.70       8/9              0/9
```

At 0.45 you discard a third of the correct answers to remove two-thirds of the
distractors; by 0.55, where 8/9 answers survive, only 2/9 distractors are still being cut.
There is no cutoff that is clearly better than not having one.

The reason is that absolute cosine distance here tracks **question–passage phrasing
similarity**, not correctness: a verbose FAQ section that restates the question's wording
scores closer than the terse clause that actually answers it. Distance is meaningful for
*ranking within one query*, never as a cross-query quality scale.

A threshold would also be actively harmful: silently returning fewer than k chunks makes
retrieval failures look like sparse documentation, and the failure it is meant to
prevent — junk reaching the customer — is **already covered twice** by `generate`'s
`INSUFFICIENT_CONTEXT` sentinel and the `verify` groundedness gate. Adding a third,
demonstrably unreliable filter upstream trades a working guarantee for a heuristic that
would, on this evidence, drop correct answers first.

**Decision: top-k only.** Scores are still returned to callers (and logged) so relevance
stays observable and this can be revisited on real traffic — but nothing filters on them.

## 5. Question (c) — the drift guard and its cache

Invariant **I-11**. `retriever.py` is the only place the store is ever compared against
`settings.py`, so this check is the whole enforcement.

Why the two fields are treated differently:

* **`embedding_model` → raises.** The query is embedded *at query time*. If the store was
  built with a different model, query and document vectors are not comparable. When the
  dimensions happen to match (e.g. two 1536-dim models) pgvector returns arbitrary chunks
  **with no error**, and every downstream gate including `verify` still passes — the judge
  is asked "is this draft grounded in this context?" and the draft *is* grounded in the
  garbage it was given. Silent, total corruption. Nothing else catches it.
* **`chunk_size` / `chunk_overlap` → warn.** Nothing in the retrieval path reads them; a
  mismatch means the store predates a settings change, so chunks are *stale*, not *wrong*.
  Kept because the realistic failure is a half-finished edit (settings tuned, ingestion not
  re-run), and the probe chunk is already fetched for the embedding check — so it costs
  nothing.

**Caching shape (c):** the probe is one similarity search per collection. Uncached it would
run on *every* retrieval — doubling query latency and OpenAI spend forever, to re-answer a
question whose answer cannot change while the process lives. Cached per `(collection,
embedding_model, chunk_size, chunk_overlap)` via `functools.cache`: one probe per
collection per process, and the settings in the key mean a notebook that mutates settings
re-probes instead of returning a stale pass.

In [8]:
import warnings
from functools import cache


class StoreConfigMismatch(RuntimeError):
    """The store was built with different settings than the ones now configured."""


@cache
def assert_store_matches(
    product_id: str, embedding_model: str, chunk_size: int, chunk_overlap: int
) -> None:
    """Compare one collection's stored build config against current settings.

    Build-config values are cache-key parameters rather than reads of `settings` inside the
    body, so that changing settings invalidates the cache instead of returning a stale pass.
    """
    collection = registry.get(product_id).doc_collection
    probe = open_store(product_id).similarity_search("a", k=1)
    if not probe:
        raise StoreConfigMismatch(
            f"collection '{collection}' is empty — run: "
            f"python -m app.retrieval.ingestion --product {product_id}"
        )

    meta = probe[0].metadata
    built_with = meta.get("embedding_model")
    if built_with != embedding_model:
        raise StoreConfigMismatch(
            f"collection '{collection}' was built with embedding model '{built_with}', but "
            f"settings specify '{embedding_model}'. Vectors from different models are not "
            f"comparable — re-run: python -m app.retrieval.ingestion"
        )

    for field, current in (("chunk_size", chunk_size), ("chunk_overlap", chunk_overlap)):
        if meta.get(field) != current:
            warnings.warn(
                f"collection '{collection}' was built with {field}={meta.get(field)}, but "
                f"settings specify {current}. Chunks are stale (not wrong) — re-run "
                f"ingestion to sync.",
                stacklevel=2,
            )


for pid in registry.product_ids():
    assert_store_matches(pid, settings.EMBEDDING_MODEL, settings.CHUNK_SIZE, settings.CHUNK_OVERLAP)
print("OK — all 4 collections match current settings")

OK — all 4 collections match current settings


In [9]:
# The guard must actually fire. Both branches, against the real store.
try:
    assert_store_matches("digital_fd", "text-embedding-3-large", settings.CHUNK_SIZE, settings.CHUNK_OVERLAP)
    print("FAIL — embedding-model mismatch did not raise")
except StoreConfigMismatch as exc:
    print("OK raises on embedding-model mismatch:\n   ", exc)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    assert_store_matches("digital_fd", settings.EMBEDDING_MODEL, 500, 50)
    assert caught, "chunk-param mismatch should warn"
    print("\nOK warns on chunk-param mismatch (does not raise):\n   ", caught[0].message)

OK raises on embedding-model mismatch:
    collection 'fd_docs' was built with embedding model 'text-embedding-3-small', but settings specify 'text-embedding-3-large'. Vectors from different models are not comparable — re-run: python -m app.retrieval.ingestion



OK warns on chunk-param mismatch (does not raise):
    collection 'fd_docs' was built with chunk_size=1000, but settings specify 500. Chunks are stale (not wrong) — re-run ingestion to sync.


In [10]:
# Cache shape: one probe per collection per process, regardless of call count.
# (Measured on the v2 guard in §5b — this cell keeps the prototype's numbers for contrast.)
assert_store_matches.cache_clear()
for _ in range(25):
    for pid in registry.product_ids():
        assert_store_matches(pid, settings.EMBEDDING_MODEL, settings.CHUNK_SIZE, settings.CHUNK_OVERLAP)

info = assert_store_matches.cache_info()
print(info)
assert info.misses == len(registry.product_ids()), "expected exactly one probe per collection"
print(f"OK — 100 calls, {info.misses} actual probes (one per collection), {info.hits} cached")

CacheInfo(hits=96, misses=4, maxsize=None, currsize=4)
OK — 100 calls, 4 actual probes (one per collection), 96 cached


## 5b. The guard has two mismatch shapes — and the prototype only survives one

The §5 prototype probes with `similarity_search("a", k=1)`, which **embeds a query before
it can compare anything**. That hides a split in the failure it is meant to catch:

| store built with | settings specify | dimensions | what actually happens |
|---|---|---|---|
| `3-small` (1536) | `3-large` (3072) | differ | pgvector raises `DataError` **at the probe** |
| `3-small` (1536) | another 1536-dim model | match | no error — the string compare is the only thing that catches it |

Row 2 is the invariant's real target: silent corruption, arbitrary chunks, every
downstream gate still passing. Row 1 is *safe* (nothing silent about it) but surfaces as a
raw SQLAlchemy `DataError` rather than `StoreConfigMismatch` — so a caller catching the
guard's own exception misses it, and the operator gets a vector-dimension stack trace
instead of "re-run ingestion".

The §5 cell only ever tested row 1 **by accident**: its `open_store()` always built the
embedder from `settings.EMBEDDING_MODEL`, so the `embedding_model` argument reached the
string comparison without ever being used to embed. A port that threads the model through
to the embedder — the natural way to write it — changes which exception escapes.

Both rows are fixed by not embedding at all: the build-config metadata is a plain column
read.

In [11]:
# Demonstrate the split. A guard that embeds with the *candidate* model (what a natural
# port does) raises DataError, not StoreConfigMismatch, when dimensions differ.
import sqlalchemy as sa


def _probe_by_embedding(product_id: str, embedding_model: str):
    """The §5 shape, but honestly threading the model through to the embedder."""
    store = PGVector(
        embeddings=OpenAIEmbeddings(model=embedding_model),
        collection_name=registry.get(product_id).doc_collection,
        connection=settings.DATABASE_URL,
    )
    return store.similarity_search("a", k=1)


try:
    _probe_by_embedding("digital_fd", "text-embedding-3-large")   # 3072-dim vs 1536 store
    print("no error")
except StoreConfigMismatch:
    print("StoreConfigMismatch (what callers expect)")
except Exception as exc:
    print(f"LEAKS {type(exc).__name__}: {str(exc).splitlines()[0][:100]}")
    print("   -> a caller catching StoreConfigMismatch does NOT catch this")

LEAKS DataError: (psycopg.errors.DataException) different vector dimensions 1536 and 3072
   -> a caller catching StoreConfigMismatch does NOT catch this


In [12]:
# The fix: read build-config metadata with a plain column lookup — no embedding call, so
# no vector ever gets built and the dimension error is unreachable.
_engine = sa.create_engine(settings.DATABASE_URL)

_PROBE_SQL = sa.text("""
    SELECT e.cmetadata
    FROM langchain_pg_embedding e
    JOIN langchain_pg_collection c ON e.collection_id = c.uuid
    WHERE c.name = :collection
    LIMIT 1
""")


def read_build_config(collection: str) -> dict | None:
    """One chunk's metadata, or None if the collection is absent/empty."""
    with _engine.connect() as conn:
        row = conn.execute(_PROBE_SQL, {"collection": collection}).fetchone()
    return row[0] if row else None


for pid in registry.product_ids():
    meta = read_build_config(registry.get(pid).doc_collection)
    print(f"{pid:14} model={meta['embedding_model']:24} "
          f"chunk={meta['chunk_size']}/{meta['chunk_overlap']}")

digital_fd     model=text-embedding-3-small   chunk=1000/150
digital_gold   model=text-embedding-3-small   chunk=1000/150
bonds          model=text-embedding-3-small   chunk=1000/150
mutual_funds   model=text-embedding-3-small   chunk=1000/150


In [13]:
# v2 of the guard — metadata-first. This is the version that ports.
@cache
def assert_store_matches_v2(
    product_id: str, embedding_model: str, chunk_size: int, chunk_overlap: int
) -> None:
    collection = registry.get(product_id).doc_collection
    meta = read_build_config(collection)
    if meta is None:
        raise StoreConfigMismatch(
            f"collection '{collection}' is empty or does not exist — run: "
            f"python -m app.retrieval.ingestion --product {product_id}"
        )

    built_with = meta.get("embedding_model")
    if built_with != embedding_model:
        raise StoreConfigMismatch(
            f"collection '{collection}' was built with embedding model '{built_with}', but "
            f"settings specify '{embedding_model}'. Vectors from different models are not "
            f"comparable — re-run: python -m app.retrieval.ingestion"
        )

    for field, current in (("chunk_size", chunk_size), ("chunk_overlap", chunk_overlap)):
        if meta.get(field) != current:
            warnings.warn(
                f"collection '{collection}' was built with {field}={meta.get(field)}, but "
                f"settings specify {current}. Chunks are stale (not wrong) — re-run "
                f"ingestion to sync.",
                stacklevel=2,
            )


# Both mismatch shapes must now raise StoreConfigMismatch, not DataError.
CASES = [
    ("dimension MISMATCH (3072 vs 1536)", "text-embedding-3-large"),
    ("dimension MATCH   (1536 vs 1536)", "text-embedding-ada-002"),
]
for label, model in CASES:
    try:
        assert_store_matches_v2("digital_fd", model, settings.CHUNK_SIZE, settings.CHUNK_OVERLAP)
        print(f"FAIL {label}: did not raise")
    except StoreConfigMismatch:
        print(f"OK   {label}: StoreConfigMismatch")
    except Exception as exc:
        print(f"FAIL {label}: leaked {type(exc).__name__}")

# Happy path + the warn branch still behave.
for pid in registry.product_ids():
    assert_store_matches_v2(pid, settings.EMBEDDING_MODEL, settings.CHUNK_SIZE, settings.CHUNK_OVERLAP)
print("\nOK   all 4 collections pass against current settings")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    assert_store_matches_v2("digital_fd", settings.EMBEDDING_MODEL, 500, 50)
    assert caught, "chunk-param mismatch should warn, not raise"
print("OK   chunk-param mismatch still warns:", str(caught[0].message)[:70], "...")

OK   dimension MISMATCH (3072 vs 1536): StoreConfigMismatch
OK   dimension MATCH   (1536 vs 1536): StoreConfigMismatch



OK   all 4 collections pass against current settings
OK   chunk-param mismatch still warns: collection 'fd_docs' was built with chunk_size=1000, but settings spec ...


### Finding (5b) — the guard reads metadata, it does not embed

`assert_store_matches_v2` replaces the §5 prototype. A plain column read on
`langchain_pg_embedding.cmetadata`, joined to the collection by name:

* **Both mismatch shapes raise `StoreConfigMismatch`.** No query vector is ever built, so
  the dimension `DataError` is unreachable and callers have one exception to catch.
* **The check runs before any embedding spend**, instead of requiring a live OpenAI call to
  discover that the store is unusable.
* **Startup gets cheaper** — the per-collection probe was the only OpenAI call the guard
  made, and it is now pure SQL.

Cost: it reaches past the `PGVector` abstraction into langchain-postgres' table layout
(`langchain_pg_embedding` / `langchain_pg_collection`). That is a real coupling and it is
worth naming — but ingestion already depends on the same library writing those tables, and
a schema change would surface immediately and loudly here rather than silently. Acceptable
for the one check whose entire job is catching a failure nothing else can see.

**This is what ports.** The §5 cell stays above as the prototype it was, and as the record
of why the metadata-first shape is the one that survives contact with a real port.

## 6. The retrieval function

Everything settled above, assembled into the shape that ports to `app/`. `retrieve()` is
guarded — the drift check runs on the first call for a collection, so no caller can query
a mismatched store even if it forgets to check.

In [14]:
def retrieve(query: str, product_id: str, k: int | None = None):
    """Top-k chunks for one product, scored, with metadata intact.

    The drift guard runs first (cached, metadata-only — see §5b), so retrieval cannot
    silently run against a store built by a different embedding model, and the check costs
    no embedding call.
    """
    assert_store_matches_v2(
        product_id, settings.EMBEDDING_MODEL, settings.CHUNK_SIZE, settings.CHUNK_OVERLAP
    )
    return open_store(product_id).similarity_search_with_score(query, k=k or settings.TOP_K)


for doc, score in retrieve("What is the penalty for breaking an FD early?", "digital_fd"):
    m = doc.metadata
    print(f"[{score:.3f}] {m['source']} (updated {m['doc_version']})")
    print("   ", doc.page_content[:150].replace("\n", " "), "...\n")

[0.475] digital_fixed_deposits_kb.docx (updated July 2026)
    5. Premature withdrawal (breaking an FD)  Retail FDs (₹1 crore and below) always carry a premature-withdrawal facility per RBI rules.  On early closur ...

[0.584] digital_fixed_deposits_kb.docx (updated July 2026)
    10. Frequently asked questions  Is my FD safe if the bank fails? Up to ₹5 lakh (principal + interest, per bank, per depositor) is protected by DICGC.  ...

[0.631] digital_fixed_deposits_kb.docx (updated July 2026)
    Is a digital FD different from a branch FD? No — same product, same regulation, same DICGC insurance. Only the channel differs. ...

[0.633] digital_fixed_deposits_kb.docx (updated July 2026)
    8. Taxation of FD interest (FY 2026-27)  FD interest is fully taxable as “Income from Other Sources” at the depositor’s income-tax slab rate. Tax is d ...



In [15]:
# Scoping: a query about one product must never return another product's chunks.
# This is what makes registry-driven collection routing meaningful.
FULL_META = {"product", "source", "doc_version", "ingested_at", "content_hash",
             "embedding_model", "chunk_size", "chunk_overlap"}

for pid in registry.product_ids():
    results = retrieve("investment returns and charges", pid, k=8)
    assert results, f"{pid}: retrieval returned nothing"
    for doc, _ in results:
        assert doc.metadata["product"] == pid, f"cross-product leak in {pid}!"
        missing = FULL_META - doc.metadata.keys()
        assert not missing, f"{pid}: chunk lost metadata {missing}"
    print(f"OK {pid:14} {len(results)} chunks, all scoped, 8/8 metadata fields intact")

OK digital_fd     8 chunks, all scoped, 8/8 metadata fields intact


OK digital_gold   8 chunks, all scoped, 8/8 metadata fields intact


OK bonds          8 chunks, all scoped, 8/8 metadata fields intact


OK mutual_funds   8 chunks, all scoped, 8/8 metadata fields intact


In [16]:
# Unknown product must fail loudly at the registry, before any embedding spend (I-8):
# an LLM-invented product label cannot silently resolve to some default collection.
try:
    retrieve("anything", "crypto")
    print("FAIL — unknown product did not raise")
except KeyError as exc:
    print("OK unknown product rejected:", exc)

OK unknown product rejected: "unknown product 'crypto'; known products: ['digital_fd', 'digital_gold', 'bonds', 'mutual_funds']"


## 7. Settled — port this

| Question | Decision | Why |
|---|---|---|
| (a) `TOP_K` | **4, unchanged** | 8/9 probes answer at rank ≤3. k=5 only rescues the known mutual-funds KB gap, at the cost of a 5th noisy chunk on every other query. Fix the doc, not k. |
| (b) threshold | **None — top-k only** | Correct (0.332–0.709) and distractor (0.335–0.666) distances are near-coincident; in 3/9 probes the correct chunk scores *worse* than the best distractor. No cutoff beats having none. `INSUFFICIENT_CONTEXT` + `verify` already cover the failure. Scores returned for observability, never filtered on. |
| (c) cache | **`functools.cache` keyed on (product_id, model, size, overlap)** | One probe per collection per process (measured: 100 calls → 4 probes). Settings in the key so a settings change re-probes rather than returning a stale pass. |
| (d) guard probe | **Metadata-only SQL read, not `similarity_search`** (§5b) | The embedding-based prototype leaks `DataError` instead of `StoreConfigMismatch` when the candidate model's dimensions differ from the store's. Reading `cmetadata` directly makes both mismatch shapes raise the same exception, and removes the guard's only OpenAI call. |

**Port to `app/retrieval/retriever.py`:** `StoreConfigMismatch`, `read_build_config`,
`assert_store_matches_v2` (as `assert_store_matches`), `retrieve`, plus a module-level
store cache. No new constants — `TOP_K`, `EMBEDDING_MODEL`, `CHUNK_SIZE`, `CHUNK_OVERLAP`
all already live in `settings.py`.

**Port check — the notebook cannot prove this on its own.** §5b exists because a first port
attempt threaded `embedding_model` through to the embedder and changed which exception
escaped, while the notebook still passed. So the ported module must be re-tested for
*both* mismatch shapes (dimensions differing **and** dimensions matching), not just
re-imported and smoke-tested. That belongs in `tests/unit/` (row 1.7) and in
`v1_module_tests.ipynb`.

**Follow-ups (not 1.5):** the mutual-funds NAV doc gap stays open; the probe set in §2 is
good raw material for `tests/eval/`.